In [1]:
import csv
import json


INPUT_CSV = "biodiversity_funding_full_cleaned.csv"       # Replace with your CSV file
OUTPUT_JSONL = "funding_finetune_data.jsonl" # The file we'll create


In [2]:

def create_qa_pairs(row):
    """
    Given one CSV row (as a dict), return multiple prompt-completion pairs.
    Each pair references columns from the row.
    """

    # Pull the columns (use .get() to avoid KeyErrors if a column is missing)
    funder = row.get("Funder", "")
    funding_amount = row.get("Funding Amount", "")
    application_deadline = row.get("Application Deadline", "")
    funding_focus = row.get("Funding Focus", "")
    eligibility = row.get("Eligibility Criteria", "")
    application_process = row.get("Application Process", "")
    past_recipients = row.get("Past Recipients (if available)", "")
    contact_info = row.get("Contact Information", "")
    geographic_focus = row.get("Geographic Focus", "")
    gotchas = row.get("Gotchas", "")
    additional_notes = row.get("Additional Notes", "")

    # We will create a few Q/A pairs per row:
    qa_list = []

    # Q/A #1: Basic overview
    prompt_1 = (
        f"I need a grant or funding that supports {funding_focus}. "
        f"Does {funder} provide this kind of funding?"
    )
    completion_1 = (
        f"{funder} offers {funding_amount}. Their application deadline is {application_deadline}. "
        f"Eligibility: {eligibility}. Application process: {application_process}."
    )
    qa_list.append({"prompt": prompt_1, "completion": completion_1})

    # Q/A #2: Gotchas or special conditions
    prompt_2 = (
        f"Are there any special conditions or gotchas when applying to {funder}?"
    )
    completion_2 = (
        f"For {funder}, gotchas include: {gotchas}. "
        f"Additional notes: {additional_notes}."
    )
    qa_list.append({"prompt": prompt_2, "completion": completion_2})

    # Q/A #3: Past recipients, if available
    if past_recipients.strip():
        prompt_3 = f"Any past recipients for {funder}?"
        completion_3 = (
            f"Past recipients include: {past_recipients}. For more info, contact {contact_info}."
        )
        qa_list.append({"prompt": prompt_3, "completion": completion_3})

    # Q/A #4: Geographic focus, if not empty
    if geographic_focus.strip():
        prompt_4 = f"Does {funder} support projects in {geographic_focus}?"
        completion_4 = (
            f"{funder} focuses on {geographic_focus}. "
            f"Feel free to reach out via {contact_info} for more details."
        )
        qa_list.append({"prompt": prompt_4, "completion": completion_4})

    return qa_list



In [8]:
def main():
    with open(INPUT_CSV, "r", encoding="utf-8") as infile, \
         open(OUTPUT_JSONL, "w", encoding="utf-8") as outfile:
        
        reader = csv.DictReader(infile)
        
        for row in reader:

            qa_pairs = create_qa_pairs(row)
     
            
            # Write each Q/A pair as one JSON record per line
            for qa in qa_pairs:
                record = {
                    "prompt": qa["prompt"],
                    "completion": qa["completion"]
                }
                # Convert record to JSON, write it out
                outfile.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f"Done! Created JSONL file: {OUTPUT_JSONL}")

if __name__ == "__main__":
    main()


Done! Created JSONL file: funding_finetune_data.jsonl


In [9]:
import json
import random

# File paths (update these to match your setup)
INPUT_FILE = "funding_finetune_data.jsonl"     # Your full dataset
TRAIN_FILE = "funding_train.jsonl"             # Output: training subset
VAL_FILE = "funding_val.jsonl"                 # Output: validation subset

# Ratio of data to use for training
TRAIN_RATIO = 0.8

def main():
    # 1) Read all lines from the JSONL
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        lines = f.readlines()

    # 2) Shuffle the lines
    random.shuffle(lines)

    # 3) Compute split index
    total_lines = len(lines)
    train_count = int(total_lines * TRAIN_RATIO)

    # 4) Split into train and validation
    train_lines = lines[:train_count]
    val_lines = lines[train_count:]

    # 5) Write out the new files
    with open(TRAIN_FILE, "w", encoding="utf-8") as f_train:
        for line in train_lines:
            f_train.write(line)

    with open(VAL_FILE, "w", encoding="utf-8") as f_val:
        for line in val_lines:
            f_val.write(line)

    print(f"Total lines: {total_lines}")
    print(f"Training lines: {len(train_lines)} -> {TRAIN_FILE}")
    print(f"Validation lines: {len(val_lines)} -> {VAL_FILE}")

if __name__ == "__main__":
    main()


Total lines: 352
Training lines: 281 -> funding_train.jsonl
Validation lines: 71 -> funding_val.jsonl


In [6]:
# The JSONL file is essentially a list of user-question → ideal-answer examples covering your funding data. Each line in that file has:

# A “prompt” (the user’s hypothetical query), such as “Which grants offer marine biodiversity funding?”

# A “completion” (the correct or desired reply), referencing columns like Funding Amount, Application Deadline, Eligibility Criteria, etc.

# So, intuitively, the JSONL file is a set of labeled examples (prompts and completions) that show the LLM how to respond to real-world questions about your grants.

In [7]:
# In the JSONL generator code, we used a template-based or scripted prompting approach rather than an LLM-based prompt. Essentially, the code hard-codes certain question–answer formats to create “synthetic” training examples, referencing the CSV columns. Here’s how it works in detail:

# Template (Manual) Q/A Generation

# For each row in the CSV, the code manually constructs a question (“prompt”) and a matching answer (“completion”) by inserting the row’s column values into a few prewritten sentences.

# For example:

# python
# Copy
# Edit
# # Q/A #1: Basic overview
# prompt_1 = f"We are looking for funding focused on {funding_focus}. Does {funder} fit our needs?"
# completion_1 = f"{funder} offers {funding_amount}. The application deadline is {application_deadline}. ..."
# Notice how the prompt references the “funder” and “funding_focus” columns, while the completion includes “funding_amount,” “application_deadline,” etc.

# No LLM Generation

# The script does not ask an LLM to generate these training examples. Instead, it uses static string templates in Python.

# This means the code is effectively a “prompt engineering” shortcut—it automatically produces question–answer pairs that embed the CSV data in predictable ways.

# Fixed Questions

# In the example code, you see about 3–4 Q/A pairs per row:

# One about the general overview (focus, amount, deadline)

# One about special “gotchas” or notes

# Optional ones for “past recipients” or “geographic focus,” if they’re not empty in the CSV

# These are hard-coded in the function create_qa_pairs(row).

Next steps -

1) Reviewing the Training & Validation Datasets
Before kicking off fine-tuning, it’s important to validate the datasets. Here’s how your teammates can do it:

A. Format & Structural Review
Check JSONL Integrity

Ensure each line in funding_train.jsonl and funding_val.jsonl is valid JSON (no malformed lines, trailing commas, or broken braces).

Quick approach: use a script or command-line JSON validator (e.g., jq . funding_train.jsonl on Linux) to confirm each line is parseable.

Count Lines

Verify the correct split ratio (e.g., 80/20). If you expected 80% for training, confirm the line count ratio roughly matches.

Spot Check a Few Entries

Open each file in a text editor. Check about 5–10 random lines to see if the "prompt" and "completion" fields have valid strings, referencing the correct columns.

B. Content Accuracy Review
Map a Prompt–Completion Pair

For a random line in funding_train.jsonl, locate the corresponding row in the original CSV (if possible) to confirm that the “completion” references the correct funder, deadline, eligibility, etc.

Look for Inconsistencies

Do any completions mention a “Rolling Deadline” that doesn’t appear in the CSV row?

Are there placeholders like “No data available” that might need to be replaced or removed?

Check Column Coverage

Make sure your Q/A pairs actually use the columns you intended (Funder, Funding Amount, Deadline, etc.). If “Gotchas” is always empty, confirm that’s really the case in the original CSV or if it’s missing data.

Language & Style

The dataset is likely “template-based,” so each line may look similar. That’s okay, but ensure it’s still readable. If the style is too terse or too verbose, you can refine before training.

C. Approve for Fine-Tuning
After your teammates confirm the dataset is correct, they should sign off that funding_train.jsonl and funding_val.jsonl are ready for model training.

2) Steps for LLM Fine-Tuning (e.g., LLaMA or Another Open-Source Model)
Now that the data is ready, your teammates can proceed with fine-tuning. Below are generic steps tailored to open-source LLMs like LLaMA or GPT-Neo, which can be adapted to different frameworks (e.g., Hugging Face Transformers).


Download/Load Model Weights

Acquire the LLaMA or other open-source checkpoint. If it’s a Meta-licensed model, you’ll need appropriate permissions.

For example, you might store them locally or use the Hugging Face Hub (some models are restricted or require an agreement).

B. Data Preparation in Hugging Face Format
You can either:

Directly use your JSONL files with a custom dataset loader.


from datasets import load_dataset

# If you have a JSONL with "prompt", "completion" fields:
dataset = load_dataset("json", data_files={"train": "funding_train.jsonl", 
                                           "validation": "funding_val.jsonl"})
C. Model Configuration
Choose Fine-Tuning Method

Full fine-tuning (requires more GPU memory).

LoRA / PEFT (Parameter-Efficient Fine-Tuning) to reduce memory usage and training time. This is popular for LLaMA-based models.

Tokenization

Use the same tokenizer that the base model used (e.g., LLaMA tokenizer).

Make sure to handle the special tokens for start/end of sequence, new lines, etc.

Hyperparameters

Common defaults:

Learning rate: ~1e-5 to 2e-5 for full fine-tuning; often 1e-4 to 2e-4 for LoRA.

Batch size: depends on GPU memory (could be 8, 16, or 32 tokens).

Epochs: 3–5 is common, but it varies.

Trainer (Hugging Face) or Custom Training Loop

If you use Hugging Face’s Trainer or Accelerate, set up a TrainingArguments object.


from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./llama-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    logging_steps=100,
    evaluation_strategy="steps",
    save_steps=500,
    learning_rate=1e-4
)
D. Launch the Fine-Tuning
Initialize the Model

from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("path/to/llama-checkpoint")
tokenizer = AutoTokenizer.from_pretrained("path/to/llama-checkpoint")
LoRA / PEFT Setup (Optional)

If you choose LoRA, wrap the model with peft library calls to reduce training overhead.

Train

python
Copy
Edit
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer
)
trainer.train()
Monitor Logs

Check training and validation loss in real-time. If validation loss stops improving or starts increasing, you might be overfitting.

Save the Final Model

The trainer can automatically save checkpoints.

You’ll get a folder with your fine-tuned weights.

E. Evaluate and Test the Fine-Tuned Model
Use the Validation Set

Evaluate the final model’s performance. Check how it handles queries not in the training set.

Real Queries

If possible, run actual user queries (like “Which funder has a rolling deadline for coastal projects over $100k?”).

See if the model references the correct data from the JSONL.

Look for Hallucinations

Does it invent deadlines or amounts not in the CSV? If so, you may need to incorporate a retrieval step or more training examples.

